In [7]:
import numpy as np
import pickle
import os
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy
import matplotlib.pyplot as plt

In [2]:
save_path = "outputs/checkpoints"
dataset_path = "outputs/datasets"
os.makedirs(dataset_path, exist_ok=True)

In [ ]:

def collect_trajectory(model, env, seed=None):
    """Collect a single trajectory with a specific environment seed."""
    obs, _ = env.reset(seed=seed)
    trajectory = []
    total_reward = 0
    done = False
    while not done:
        # Note: Added deterministic=False to allow for more variety in trajectories
        action, _ = model.predict(obs, deterministic=False)
        next_obs, reward, terminated, truncated, _ = env.step(action)
        trajectory.append((obs, action, reward))
        total_reward += reward
        obs = next_obs
        done = terminated or truncated
    return trajectory, total_reward

def compute_preference_label(R1, R2):
    """Sample preference label using Bradley-Terry model with log-sum-exp trick."""
    c = max(R1, R2)
    p = np.exp(R1 - c) / (np.exp(R1 - c) + np.exp(R2 - c))
    label = np.random.choice([0, 1], p=[p, 1 - p])  # 0 = tau_1 preferred, 1 = tau_2 preferred
    return label

def generate_dataset_with_seed(pi1, pi2, env, K, dataset_seed):
    """
    Generates K pairs and bundles metadata for Part 3.
    """
    # Fix the seed for the probabilistic labeling
    np.random.seed(dataset_seed)
    
    # We will generate a unique env seed for every pair
    # but use the same env seed for both tau1 and tau2 within that pair.
    # This is called 'Common Random Numbers' and reduces variance.
    rng = np.random.default_rng(dataset_seed)
    env_seeds = rng.integers(0, 1000000, size=K)
    
    dataset = []
    for i in range(K):
        current_env_seed = int(env_seeds[i])
        
        tau1, R1 = collect_trajectory(pi1, env, seed=current_env_seed)
        tau2, R2 = collect_trajectory(pi2, env, seed=current_env_seed)
        
        label = compute_preference_label(R1, R2)
        
        dataset.append({
            "tau_1": tau1,
            "tau_2": tau2,
            "R_1": R1,
            "R_2": R2,
            "label": label
        })
    
    return dataset

# CartPole datasets

In [ ]:

K_values = [50, 200, 500, 1000]
seed = 1
env = gym.make("CartPole-v1")

print(f"\n=== Generating datasets for CartPole seed {seed} ===")

pi1 = PPO.load(os.path.join(save_path, f"pi1_cartpole_seed{seed}"))
pi2 = PPO.load(os.path.join(save_path, f"pi2_cartpole_seed{seed}"))

# sanity check: verify policies produce expected rewards
mean_r1, std_r1 = evaluate_policy(pi1, env, n_eval_episodes=20, deterministic=True)
mean_r2, std_r2 = evaluate_policy(pi2, env, n_eval_episodes=20, deterministic=True)
print(f"pi1 reward: {mean_r1:.1f} +/- {std_r1:.1f}")
print(f"pi2 reward: {mean_r2:.1f} +/- {std_r2:.1f}")

# compute expected label rate analytically
c = max(mean_r1, mean_r2)
expected_p = np.exp(mean_r1 - c) / (np.exp(mean_r1 - c) + np.exp(mean_r2 - c))
print(f"Expected % of pairs preferring pi1: {100*expected_p:.1f}%")

# generate largest dataset for visualization
dataset_1000 = generate_dataset_with_seed(pi1, pi2, env, 1000, dataset_seed=seed)

R1_list = [d["R_1"] for d in dataset_1000]
R2_list = [d["R_2"] for d in dataset_1000]
labels   = [d["label"] for d in dataset_1000]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle(f"CartPole seed {seed} --- dataset sanity check (K=1000)")

# plot 1: reward distributions
axes[0].hist(R1_list, bins=30, alpha=0.6, label="pi1 (R1)", color="green")
axes[0].hist(R2_list, bins=30, alpha=0.6, label="pi2 (R2)", color="red")
axes[0].set_xlabel("Total episode reward")
axes[0].set_ylabel("Count")
axes[0].set_title("Reward distributions")
axes[0].legend()

# plot 2: label distribution
preferred_tau1 = labels.count(0)
preferred_tau2 = labels.count(1)
axes[1].bar(["tau1 preferred (pi1)", "tau2 preferred (pi2)"],
            [preferred_tau1, preferred_tau2],
            color=["green", "red"], alpha=0.7)
axes[1].set_ylabel("Count")
axes[1].set_title(f"Label distribution\n(expected {100*expected_p:.1f}% prefer tau1)")
for i, v in enumerate([preferred_tau1, preferred_tau2]):
    axes[1].text(i, v + 5, f"{100*v/1000:.1f}%", ha="center")

plt.tight_layout()
plt.show()

# now generate and save all dataset sizes
for K in K_values:
    dataset = generate_dataset_with_seed(pi1, pi2, env, K, dataset_seed=seed)
    preferred_tau1 = sum(1 for d in dataset if d["label"] == 0)
    mean_R1 = np.mean([d["R_1"] for d in dataset])
    mean_R2 = np.mean([d["R_2"] for d in dataset])
    print(f"K={K}: {preferred_tau1}/{K} pairs prefer tau1 ({100*preferred_tau1/K:.1f}%) | "
            f"avg R1={mean_R1:.1f}, avg R2={mean_R2:.1f}")
    
    filename = os.path.join(dataset_path, f"cartpole_seed{seed}_K{K}.pkl")
    with open(filename, "wb") as f:
        pickle.dump(dataset, f)
    print(f"Saved: {filename}")

print("\nDone! All datasets generated.")

# Pendulum datasets

In [ ]:
K_values = [50, 200, 500, 1000]
seed = 1
env = gym.make("Pendulum-v1")


print(f"\n=== Generating datasets for Pendulum seed {seed} ===")
pi1 = PPO.load(os.path.join(save_path, f"pi1_pendulum_seed{seed}"))
pi2 = PPO.load(os.path.join(save_path, f"pi2_pendulum_seed{seed}"))

# sanity check
mean_r1, std_r1 = evaluate_policy(pi1, env, n_eval_episodes=20, deterministic=True)
mean_r2, std_r2 = evaluate_policy(pi2, env, n_eval_episodes=20, deterministic=True)
print(f"pi1 reward: {mean_r1:.1f} +/- {std_r1:.1f}")
print(f"pi2 reward: {mean_r2:.1f} +/- {std_r2:.1f}")

c = max(mean_r1, mean_r2)
expected_p = np.exp(mean_r1 - c) / (np.exp(mean_r1 - c) + np.exp(mean_r2 - c))
print(f"Expected % of pairs preferring pi1: {100*expected_p:.1f}%")

# generate largest dataset for visualization
dataset_1000 = generate_dataset_with_seed(pi1, pi2, env, 1000, dataset_seed=seed)

R1_list = [d["R_1"] for d in dataset_1000]
R2_list = [d["R_2"] for d in dataset_1000]
labels   = [d["label"] for d in dataset_1000]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle(f"Pendulum seed {seed} --- dataset sanity check (K=1000)")

# plot 1: reward distributions
axes[0].hist(R1_list, bins=30, alpha=0.6, label="pi1 (R1)", color="green")
axes[0].hist(R2_list, bins=30, alpha=0.6, label="pi2 (R2)", color="red")
axes[0].set_xlabel("Total episode reward")
axes[0].set_ylabel("Count")
axes[0].set_title("Reward distributions")
axes[0].legend()

# plot 2: label distribution
preferred_tau1 = labels.count(0)
preferred_tau2 = labels.count(1)
axes[1].bar(["tau1 preferred (pi1)", "tau2 preferred (pi2)"],
            [preferred_tau1, preferred_tau2],
            color=["green", "red"], alpha=0.7)
axes[1].set_ylabel("Count")
axes[1].set_title(f"Label distribution\n(expected {100*expected_p:.1f}% prefer tau1)")
for i, v in enumerate([preferred_tau1, preferred_tau2]):
    axes[1].text(i, v + 5, f"{100*v/1000:.1f}%", ha="center")

plt.tight_layout()
plt.show()

# now generate and save all dataset sizes
for K in K_values:
    dataset = generate_dataset_with_seed(pi1, pi2, env, K, dataset_seed=seed)
    preferred_tau1 = sum(1 for d in dataset if d["label"] == 0)
    mean_R1 = np.mean([d["R_1"] for d in dataset])
    mean_R2 = np.mean([d["R_2"] for d in dataset])
    print(f"K={K}: {preferred_tau1}/{K} pairs prefer tau1 ({100*preferred_tau1/K:.1f}%) | "
            f"avg R1={mean_R1:.1f}, avg R2={mean_R2:.1f}")

    filename = os.path.join(dataset_path, f"pendulum_seed{seed}_K{K}.pkl")
    with open(filename, "wb") as f:
        pickle.dump(dataset, f)
    print(f"Saved: {filename}")

print("\nDone! All datasets generated.")